In [2]:
from google.colab import drive
drive.mount('/content/drive')

KeyboardInterrupt: 

In [ ]:
import torch
print("¿Tengo GPU conectada?:", torch.cuda.is_available())

¿Tengo GPU conectada?: True


In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import transformers
transformers.logging.set_verbosity_error() 
from transformers.utils import logging
logging.disable_progress_bar() 

from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

print(" Preparando el entorno con PyTorch...")

# 1. Cargar datos
df = pd.read_csv("/content/drive/MyDrive/tcga_preprocesado.csv")

X = df['text'].astype(str).tolist()
mapeo_etiquetas = {'T1': 0, 'T2': 1, 'T3': 2, 'T4': 3}
y = df['t'].map(mapeo_etiquetas).tolist()

# 2. División Train/Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 3. Cargar el Tokenizer y el Modelo Nativo
nombre_modelo = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer = AutoTokenizer.from_pretrained(nombre_modelo)
modelo_bert = AutoModelForSequenceClassification.from_pretrained(nombre_modelo, num_labels=4)

# 4. Tokenización
print(" Tokenizando textos...")
train_encodings = tokenizer(X_train, truncation=True, padding=True, max_length=512)
test_encodings = tokenizer(X_test, truncation=True, padding=True, max_length=512)

# 5. Adaptar los datos al formato exacto que pide PyTorch
class TCGA_Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TCGA_Dataset(train_encodings, y_train)
test_dataset = TCGA_Dataset(test_encodings, y_test)

# 6. Definir la métrica (Macro-F1)
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    macro_f1 = f1_score(labels, preds, average='macro')
    return {'macro_f1': macro_f1}

# 7. Configurar el entrenamiento 
training_args = TrainingArguments(
    output_dir='./resultados_bert',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",  
    logging_strategy="epoch", 
    disable_tqdm=True,        
    report_to="none"          
)

trainer = Trainer(
    model=modelo_bert,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

# 8. ¡A entrenar!
print(" Iniciando Fine-Tuning con PyTorch...")
trainer.train()

# 9. Evaluación Final
print(" Evaluando el modelo final...")
resultados = trainer.evaluate()

print("-" * 50)
print(f" Macro-F1 FINAL con ClinicalBERT (PyTorch): {resultados['eval_macro_f1']:.4f}")
print("-" * 50)

 Preparando el entorno con PyTorch...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


 Tokenizando textos...
 Iniciando Fine-Tuning con PyTorch...
{'loss': '1.1', 'grad_norm': '6.614', 'learning_rate': '3.34e-05', 'epoch': '1'}
{'eval_loss': '0.7976', 'eval_macro_f1': '0.6318', 'eval_runtime': '30.89', 'eval_samples_per_second': '33.41', 'eval_steps_per_second': '2.104', 'epoch': '1'}


In [ ]:
# Leemos el historial interno del entrenamiento para saltarnos el bug de la librería
metricas_evaluacion = [log for log in trainer.state.log_history if 'eval_macro_f1' in log]

if metricas_evaluacion:
    mejor_f1 = metricas_evaluacion[-1]['eval_macro_f1']
    print(f"🏆 Macro-F1 FINAL con ClinicalBERT (PyTorch): {mejor_f1:.4f}")
   
else:
    print("No se encontraron métricas en el historial.")

🏆 Macro-F1 FINAL con ClinicalBERT (PyTorch): 0.5539
